# 04b - IVF: sapuan nprobe di split val

Alur: set jumlah thread → muat vektor, query val, dan Tetangga exact → latih dan bangun `IndexIVFFlat` sekali → untuk setiap nprobe: cari, nilai 12 metrik, ukur p50, QPS, dan ukuran index → catat Lingkungan → tambah satu baris run per nprobe.

Keluaran:
- `outputs/tuning/runs_ivf.csv` — 7 baris per eksekusi notebook (satu per nprobe), hanya ditambah
- `outputs/tuning/environment_<env_id>.json` — catatan Lingkungan notebook ini ditambahkan

Prasyarat: `03_exact` sudah dijalankan di instance yang sama (Tetangga exact dan env_id yang sama); `pip install -r requirements.txt`.

Parameter build tetap (nlist = 4096, quantizer `IndexFlatL2`, METRIC_L2) dan parameter search nprobe disapu (K10). Pelatihan memakai sampel acak bawaan FAISS 256 · nlist = 1.048.576 vektor dengan seed k-means bawaan; jumlah sampel dan seed dicatat di params run (003a titik periksa 2a, 004a). Hanya split val yang dipakai (K2). nprobe kecil bisa menghasilkan kurang dari 5 hasil (slot −1), yang ditangani penilai sesuai K7.

Keputusan: K3, K10 (003a), K4, K7 (002a, 003a), K6 (002a), K8 (004a), H4–H6 dan cgroup (005a), pemanasan QPS (006a), K9 (003a).

In [ ]:
from pathlib import Path

# Keluaran 01_preprocessing: queries/qrels.parquet
PROCESSED_DIR = Path("../data/processed")
# Folder induk Set embedding; satu subfolder per embedding_id
EMBEDDINGS_DIR = Path("../data/embeddings")
# embedding_id yang dipakai; None = satu-satunya subfolder di EMBEDDINGS_DIR
EMBEDDING_ID = None
# Folder catatan run dan Lingkungan (K4, K8)
TUNING_DIR = Path("../outputs/tuning")
# Root project, dipakai untuk membaca total dan sisa disk
PROJECT_ROOT = Path("..")
# Nama notebook di catatan Lingkungan
NOTEBOOK_NAME = "04b_ivf"

# Algoritma di catatan run (model data: flat, hnsw, ivf, lsh)
ALGORITHM = "ivf"
# Split yang dipakai notebook ini (K2: tuning hanya di val)
SPLIT = "val"
# Jumlah hasil per query untuk semua algoritma (K7)
K = 5
# Dokumen relevan kalau relevance di qrels paling kecil nilai ini (002a)
MIN_RELEVANCE = 1

# Jumlah sel (centroid) IVF, parameter build; sekitar 3,4·√N untuk N = 1.446.315 (K10)
IVF_NLIST = 4096
# Jumlah sel yang diperiksa per query, disapu di val, satu run per nilai (K10)
IVF_NPROBE_VALUES = [1, 4, 8, 16, 32, 64, 128]

# Suku #8 dengan jarak exact sebesar ini atau lebih kecil dikeluarkan (005a, titik periksa 1a)
NEAR_ZERO_DISTANCE = 1e-3
# Jarak untuk slot −1: jarak L2 maksimum dua vektor bernorma 1 (002a)
MISSING_RESULT_DISTANCE = 2.0
# Query pemanasan p50, diambil dari awal split, tidak dihitung (005a)
LATENCY_WARMUP_QUERIES = 10
# Jumlah putaran pengukuran p50 atas semua query split (005a)
LATENCY_ROUNDS = 3
# Jumlah ulangan QPS setelah satu panggilan pemanasan T0 (005a, 006a)
QPS_REPEATS = 5

# Folder induk cgroup: v2 langsung di sini, v1 di subfolder cpu/ dan memory/ (K6, K8, 005a)
CGROUP_DIR = Path("/sys/fs/cgroup")
# Batas RAM cgroup v1 sebesar ini atau lebih dianggap tanpa batas, sama seperti "max" di v2 (005a)
MEMORY_UNLIMITED_FROM = 2**62
# Sumber model CPU dan flag AVX (K8)
CPUINFO_PATH = Path("/proc/cpuinfo")
# Batas atas jumlah thread (K6)
MAX_THREADS = 24
# Paket yang versinya ikut membentuk env_id; daftar ini sama di semua notebook (K8)
ENVIRONMENT_LIBRARIES = [
    "torch",
    "faiss-cpu",
    "sentence-transformers",
    "transformers",
    "datasets",
    "numpy",
    "pandas",
    "pyarrow",
]
# Panjang env_id: awal hash SHA-256 field statis Lingkungan, dalam karakter heksadesimal
ENV_ID_LENGTH = 16

## 1. Thread

Fungsi disalin identik dari `04a_hnsw` (K6, 005a, H18).

In [ ]:
import math
import os

import faiss
import torch


def parse_cpu_max(text: str) -> float | None:
    """Ubah isi cgroup v2 `cpu.max` menjadi jumlah CPU.

    Args:
        text: Isi berkas, contoh "400000 100000" (4 CPU) atau "max 100000" (tanpa batas).

    Returns:
        Kuota dalam satuan CPU, atau None kalau tanpa batas.
    """
    quota, period = text.split()
    return None if quota == "max" else int(quota) / int(period)


def parse_cfs_quota(quota_text: str, period_text: str) -> float | None:
    """Ubah isi cgroup v1 `cpu.cfs_quota_us` dan `cpu.cfs_period_us` menjadi jumlah CPU.

    Args:
        quota_text: Isi cpu.cfs_quota_us, contoh "400000", atau "-1" (tanpa batas).
        period_text: Isi cpu.cfs_period_us, contoh "100000".

    Returns:
        Kuota dalam satuan CPU, atau None kalau tanpa batas.
    """
    quota = int(quota_text.strip())
    return None if quota < 0 else quota / int(period_text.strip())


def fetch_cgroup_cpu_quota(cgroup_dir: Path) -> float | None:
    """Baca kuota CPU dari cgroup v2 atau v1.

    Args:
        cgroup_dir: Folder induk cgroup.

    Returns:
        Kuota dalam satuan CPU, atau None kalau tanpa batas.

    Raises:
        ValueError: Berkas v2 dan v1 sama-sama ada (host hybrid, urutan deteksi belum diputuskan, H18).
        FileNotFoundError: Berkas v2 maupun v1 tidak ada.
    """
    v2_path = cgroup_dir / "cpu.max"
    v1_quota_path = cgroup_dir / "cpu" / "cpu.cfs_quota_us"
    v1_period_path = cgroup_dir / "cpu" / "cpu.cfs_period_us"
    has_v2 = v2_path.is_file()
    has_v1 = v1_quota_path.is_file() and v1_period_path.is_file()
    if has_v2 and has_v1:
        raise ValueError(
            f"{v2_path} dan {v1_quota_path} sama-sama ada; urutan deteksi host hybrid belum diputuskan (H18)"
        )
    if has_v2:
        return parse_cpu_max(v2_path.read_text(encoding="utf-8"))
    if has_v1:
        return parse_cfs_quota(v1_quota_path.read_text(encoding="utf-8"), v1_period_path.read_text(encoding="utf-8"))
    raise FileNotFoundError(f"Tidak ada {v2_path} maupun {v1_quota_path}; butuh cgroup v1 atau v2 di Linux (H12)")


def fetch_cpu_quota(cgroup_dir: Path) -> float:
    """Baca jatah CPU efektif instance.

    Args:
        cgroup_dir: Folder induk cgroup.

    Returns:
        min(kuota cgroup, jumlah CPU sched_getaffinity), atau jumlah CPU affinity kalau kuota tanpa batas.

    Raises:
        OSError: Bukan Linux (os.sched_getaffinity tidak ada).
    """
    if not hasattr(os, "sched_getaffinity"):
        raise OSError("os.sched_getaffinity tidak tersedia; jatah CPU hanya bisa dibaca di Linux (H12)")
    cgroup_quota = fetch_cgroup_cpu_quota(cgroup_dir)
    affinity_count = float(len(os.sched_getaffinity(0)))
    return affinity_count if cgroup_quota is None else min(cgroup_quota, affinity_count)


def compute_thread_count(cpu_quota: float, max_threads: int) -> int:
    """Hitung jumlah thread dari jatah CPU (K6).

    Args:
        cpu_quota: Jatah CPU efektif.
        max_threads: Batas atas thread.

    Returns:
        ⌊min(cpu_quota, max_threads)⌋, paling sedikit 1.
    """
    return max(1, math.floor(min(cpu_quota, max_threads)))


def set_num_threads(count: int) -> None:
    """Set jumlah thread FAISS dan torch.

    Args:
        count: Jumlah thread.
    """
    faiss.omp_set_num_threads(count)
    torch.set_num_threads(count)


cpu_quota = fetch_cpu_quota(CGROUP_DIR)
thread_count = compute_thread_count(cpu_quota, MAX_THREADS)
set_num_threads(thread_count)
print({"cpu_quota": cpu_quota, "thread_count": thread_count})

## 2. Muat vektor, query val, dan Tetangga exact

Fungsi pemuat, termasuk `load_exact_neighbors`, disalin identik dari `04a_hnsw`. Tetangga exact dibaca dari keluaran `03_exact` dan disusun mengikuti urutan query val; query val yang tidak punya Tetangga exact menghentikan notebook.

In [ ]:
import json

import numpy as np
import pandas as pd


def load_parquet(path: Path) -> pd.DataFrame:
    """Baca satu tabel Parquet.

    Args:
        path: Berkas Parquet.

    Returns:
        Isi tabel.

    Raises:
        FileNotFoundError: Berkas tidak ada.
    """
    if not path.is_file():
        raise FileNotFoundError(f"Berkas tidak ditemukan: {path}")
    return pd.read_parquet(path)


def load_json(path: Path) -> dict[str, object]:
    """Baca satu berkas JSON.

    Args:
        path: Berkas JSON.

    Returns:
        Isi berkas.

    Raises:
        FileNotFoundError: Berkas tidak ada.
    """
    if not path.is_file():
        raise FileNotFoundError(f"Berkas tidak ditemukan: {path}")
    with path.open(encoding="utf-8") as file:
        return json.load(file)


def fetch_embedding_dir(embeddings_dir: Path, embedding_id: str | None) -> Path:
    """Tentukan folder Set embedding yang dipakai.

    Args:
        embeddings_dir: Folder induk Set embedding.
        embedding_id: embedding_id yang diminta, atau None untuk memakai satu-satunya subfolder.

    Returns:
        Folder Set embedding.

    Raises:
        FileNotFoundError: Folder yang diminta tidak ada, atau tidak ada subfolder sama sekali.
        ValueError: embedding_id None tetapi ada lebih dari satu subfolder.
    """
    if embedding_id is not None:
        target = embeddings_dir / embedding_id
        if not (target / "embedding_set.json").is_file():
            raise FileNotFoundError(f"Set embedding tidak ditemukan: {target}")
        return target
    candidates = sorted(path.parent for path in embeddings_dir.glob("*/embedding_set.json"))
    if not candidates:
        raise FileNotFoundError(f"Tidak ada Set embedding di {embeddings_dir}; jalankan 02_embedding dulu")
    if len(candidates) > 1:
        raise ValueError(f"Ada {len(candidates)} Set embedding di {embeddings_dir}; isi EMBEDDING_ID")
    return candidates[0]


def load_vectors(path: Path) -> np.ndarray:
    """Baca matriks vektor .npy sebagai memory map baca-saja.

    Args:
        path: Berkas .npy float32.

    Returns:
        Matriks vektor.

    Raises:
        FileNotFoundError: Berkas tidak ada.
    """
    if not path.is_file():
        raise FileNotFoundError(f"Berkas tidak ditemukan: {path}")
    return np.load(path, mmap_mode="r")


def select_split_rows(query_ids: pd.DataFrame, queries: pd.DataFrame, split: str) -> np.ndarray:
    """Cari baris vektor query yang termasuk satu split.

    Args:
        query_ids: Urutan query_id baris vektor query (query_ids.parquet).
        queries: Tabel Query dengan kolom query_id dan split.
        split: Nama split.

    Returns:
        Indeks baris vektor query untuk split itu, menaik.

    Raises:
        ValueError: Ada query_id vektor yang tidak ada di tabel Query.
    """
    split_by_id = queries.set_index("query_id")["split"]
    missing = query_ids.loc[~query_ids["query_id"].isin(split_by_id.index), "query_id"]
    if len(missing):
        raise ValueError(
            f"{len(missing)} query_id vektor tidak ada di queries.parquet, contoh: {missing.head(5).tolist()}"
        )
    splits = split_by_id.loc[query_ids["query_id"]].to_numpy()
    return np.flatnonzero(splits == split)


def build_relevance(qrels: pd.DataFrame, min_relevance: int) -> dict[str, set[str]]:
    """Kelompokkan dokumen relevan per query.

    Args:
        qrels: Tabel Penilaian relevansi.
        min_relevance: Nilai relevance terkecil yang dihitung relevan.

    Returns:
        query_id ke himpunan doc_id relevan.
    """
    relevant = qrels.loc[qrels["relevance"] >= min_relevance]
    return {query_id: set(group["doc_id"]) for query_id, group in relevant.groupby("query_id")}


def load_exact_neighbors(
    path: Path, embedding_id: str, query_ids: np.ndarray, doc_ids: np.ndarray, k: int
) -> tuple[np.ndarray, np.ndarray]:
    """Baca Tetangga exact sebagai matriks indeks dokumen dan skor, mengikuti urutan query.

    Args:
        path: exact_neighbors.parquet keluaran 03_exact.
        embedding_id: embedding_id vektor yang dicari.
        query_ids: query_id per baris hasil, sesuai urutan vektor query split.
        doc_ids: doc_id per baris vektor dokumen.
        k: Jumlah peringkat per query.

    Returns:
        Indeks baris dokumen dan skor L2² berukuran (len(query_ids), k).

    Raises:
        ValueError: embedding_id berbeda, atau ada query atau peringkat yang tidak lengkap.
    """
    neighbors = load_parquet(path)
    if not (neighbors["embedding_id"] == embedding_id).all():
        raise ValueError(f"{path} berisi embedding_id selain {embedding_id}")
    wanted = pd.MultiIndex.from_product([query_ids, range(1, k + 1)], names=["query_id", "rank"])
    ordered = neighbors.set_index(["query_id", "rank"]).reindex(wanted)
    if ordered["doc_id"].isna().any():
        missing = ordered.index[ordered["doc_id"].isna()].get_level_values("query_id").unique()
        raise ValueError(f"{len(missing)} query tanpa Tetangga exact lengkap, contoh: {list(missing[:5])}")
    row_of_doc = pd.Series(np.arange(len(doc_ids)), index=doc_ids)
    indices = row_of_doc.loc[ordered["doc_id"].to_numpy()].to_numpy().reshape(len(query_ids), k)
    scores = ordered["score"].to_numpy(dtype=np.float32).reshape(len(query_ids), k)
    return indices, scores


embedding_dir = fetch_embedding_dir(EMBEDDINGS_DIR, EMBEDDING_ID)
embedding_set = load_json(embedding_dir / "embedding_set.json")
embedding_id = embedding_set["embedding_id"]
doc_vectors = load_vectors(embedding_dir / "doc_vectors.npy")
query_vectors = load_vectors(embedding_dir / "query_vectors.npy")
doc_ids = load_parquet(embedding_dir / "doc_ids.parquet")["doc_id"].to_numpy()
query_ids_frame = load_parquet(embedding_dir / "query_ids.parquet")
split_rows = select_split_rows(query_ids_frame, load_parquet(PROCESSED_DIR / "queries.parquet"), SPLIT)
split_query_ids = query_ids_frame["query_id"].to_numpy()[split_rows]
split_vectors = np.ascontiguousarray(query_vectors[split_rows])
relevant = build_relevance(load_parquet(PROCESSED_DIR / "qrels.parquet"), MIN_RELEVANCE)
exact_indices, exact_scores = load_exact_neighbors(
    embedding_dir / "exact_neighbors.parquet", embedding_id, split_query_ids, doc_ids, K
)
print({"embedding_id": embedding_id, "documents": len(doc_ids), "split": SPLIT, "queries": len(split_query_ids)})

## 3. Latih dan bangun index IVF

`IndexIVFFlat` dengan quantizer `IndexFlatL2`, METRIC_L2, nlist = 4096 (K10). `train` dipanggil dengan seluruh vektor dokumen; FAISS sendiri mengambil sampel acak 256 · nlist = 1.048.576 vektor karena `max_points_per_centroid` = 256 tidak diubah (003a titik periksa 2a). Seed k-means dan `max_points_per_centroid` dibaca dari parameter clustering index (`index.cp`), tidak ditulis tangan (004a); kalau atributnya tidak ada, notebook berhenti. Waktu build (#11) mencakup pembuatan index, `train`, dan `add`, dan sama untuk ketujuh run.

In [ ]:
import time


def build_ivf_index(vectors: np.ndarray, nlist: int) -> tuple[faiss.IndexIVFFlat, float, dict[str, int]]:
    """Latih dan bangun IndexIVFFlat (quantizer IndexFlatL2, METRIC_L2) dan ukur waktunya.

    Args:
        vectors: Vektor dokumen float32 bernorma 1.
        nlist: Jumlah sel IVF.

    Returns:
        Index berisi semua vektor, waktu build dalam detik (#11), dan parameter pelatihan:
        seed k-means, max_points_per_centroid, dan jumlah vektor latih hasil sampel FAISS.

    Raises:
        AttributeError: Index tidak punya parameter clustering cp.seed atau cp.max_points_per_centroid.
    """
    start = time.perf_counter_ns()
    quantizer = faiss.IndexFlatL2(vectors.shape[1])
    index = faiss.IndexIVFFlat(quantizer, vectors.shape[1], nlist, faiss.METRIC_L2)
    clustering = getattr(index, "cp", None)
    if not (hasattr(clustering, "seed") and hasattr(clustering, "max_points_per_centroid")):
        raise AttributeError("IndexIVFFlat tidak punya cp.seed / cp.max_points_per_centroid; seed tidak terbaca (004a)")
    index.train(vectors)
    index.add(vectors)
    seconds = (time.perf_counter_ns() - start) / 1e9
    max_points = int(clustering.max_points_per_centroid)
    train_info = {
        "kmeans_seed": int(clustering.seed),
        "max_points_per_centroid": max_points,
        "train_points": min(len(vectors), nlist * max_points),
    }
    return index, seconds, train_info


index, build_seconds, train_info = build_ivf_index(doc_vectors, IVF_NLIST)
print({"ntotal": index.ntotal, "build_seconds": build_seconds, **train_info})

## 4. Penilai dan pengukur

Fungsi penilai, pencarian, pengukur, dan `evaluate_index` disalin identik dari `04a_hnsw` (K7, 005a, 006a). `evaluate_index` merangkai satu konfigurasi dengan urutan yang sama seperti `03_exact`: hasil yang dinilai (satu batch, tidak diukur) → p50 → QPS (T₀ lalu T₁..T₅) → #12.

In [ ]:
def search_index(index: faiss.Index, queries: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:
    """Cari top-k semua query dalam satu panggilan.

    Args:
        index: Index FAISS.
        queries: Vektor query float32.
        k: Jumlah hasil per query.

    Returns:
        Skor (L2² untuk index L2) dan indeks baris dokumen; indeks −1 untuk slot kosong.
    """
    return index.search(queries, k)


def to_doc_ids(indices: np.ndarray, doc_ids: np.ndarray) -> list[list[str | None]]:
    """Ubah indeks baris dokumen menjadi doc_id; slot −1 menjadi None.

    Args:
        indices: Indeks baris dokumen per query.
        doc_ids: doc_id per baris vektor dokumen.

    Returns:
        doc_id per query per peringkat.
    """
    safe = np.where(indices < 0, 0, indices)
    mapped = doc_ids[safe]
    return [[doc_id if index >= 0 else None for doc_id, index in zip(row, raw)] for row, raw in zip(mapped, indices)]


def to_distances(scores: np.ndarray) -> np.ndarray:
    """Ubah skor L2² FAISS menjadi jarak L2; skor negatif kecil dipotong ke 0 (005a).

    Args:
        scores: Skor L2².

    Returns:
        Jarak L2 float64.
    """
    return np.sqrt(np.maximum(scores.astype(np.float64), 0.0))


def to_result_distances(distances: np.ndarray, indices: np.ndarray, missing_distance: float) -> np.ndarray:
    """Beri jarak penalti untuk slot −1.

    Args:
        distances: Jarak L2 hasil pencarian.
        indices: Indeks hasil; −1 menandai slot kosong.
        missing_distance: Jarak untuk slot −1.

    Returns:
        Jarak dengan slot −1 diganti missing_distance.
    """
    return np.where(indices < 0, missing_distance, distances)


def _score_query(retrieved: list[str | None], relevant: set[str], k: int) -> tuple[float, ...]:
    hits = [1.0 if doc_id is not None and doc_id in relevant else 0.0 for doc_id in retrieved[:k]]
    ideal_count = min(len(relevant), k)
    dcg = sum(hit / math.log2(rank + 1) for rank, hit in enumerate(hits, start=1))
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_count + 1))
    first_hit = next((rank for rank, hit in enumerate(hits, start=1) if hit), None)
    precision_sum = sum(sum(hits[:rank]) / rank for rank, hit in enumerate(hits, start=1) if hit)
    hit_count = sum(hits)
    return (
        dcg / idcg,
        hit_count / len(relevant),
        0.0 if first_hit is None else 1.0 / first_hit,
        hit_count / k,
        precision_sum / ideal_count,
        1.0 if hit_count >= 1 else 0.0,
    )


def compute_quality_metrics(
    retrieved: list[list[str | None]], query_ids: np.ndarray, relevant: dict[str, set[str]], k: int
) -> dict[str, float]:
    """Hitung metrik kualitas #1–#6 rata-rata atas query (K7).

    Args:
        retrieved: doc_id hasil per query, None untuk slot −1.
        query_ids: query_id per baris retrieved.
        relevant: query_id ke himpunan doc_id relevan; setiap query punya paling sedikit satu.
        k: Jumlah hasil yang dinilai.

    Returns:
        nDCG@5, Recall@5, MRR@5, Precision@5, MAP@5, Hit rate@5.
    """
    names = ["ndcg_at_5", "recall_at_5", "mrr_at_5", "precision_at_5", "map_at_5", "hit_rate_at_5"]
    per_query = np.array([_score_query(row, relevant[query_id], k) for row, query_id in zip(retrieved, query_ids)])
    return dict(zip(names, per_query.mean(axis=0).tolist()))


def compute_fidelity_metrics(
    ann_indices: np.ndarray,
    ann_distances: np.ndarray,
    exact_indices: np.ndarray,
    exact_distances: np.ndarray,
    near_zero_distance: float,
) -> dict[str, float | int]:
    """Hitung k-NN Recall@5 (#7) dan Relative distance error (#8) beserta diagnostiknya (K7, 005a).

    Args:
        ann_indices: Indeks hasil algoritma; −1 untuk slot kosong.
        ann_distances: Jarak L2 hasil algoritma, slot −1 sudah diberi penalti.
        exact_indices: Indeks Tetangga exact.
        exact_distances: Jarak L2 Tetangga exact.
        near_zero_distance: Suku dengan jarak exact sebesar ini atau lebih kecil dikeluarkan.

    Returns:
        knn_recall_at_5, relative_distance_error (NaN kalau tidak ada query dengan suku tersisa),
        rde_excluded_terms, rde_queries_without_terms.
    """
    k = exact_indices.shape[1]
    matches = [
        len({index for index in ann_row if index >= 0} & set(exact_row))
        for ann_row, exact_row in zip(ann_indices, exact_indices)
    ]
    sorted_ann = np.sort(ann_distances, axis=1)
    kept = exact_distances > near_zero_distance
    safe_exact = np.where(kept, exact_distances, 1.0)
    errors = np.where(kept, (sorted_ann - exact_distances) / safe_exact, 0.0)
    kept_counts = kept.sum(axis=1)
    has_terms = kept_counts > 0
    per_query = errors[has_terms].sum(axis=1) / kept_counts[has_terms]
    return {
        "knn_recall_at_5": float(np.mean(matches)) / k,
        "relative_distance_error": float(per_query.mean()) if has_terms.any() else float("nan"),
        "rde_excluded_terms": int((k - kept_counts).sum()),
        "rde_queries_without_terms": int((~has_terms).sum()),
    }


def count_short_results(indices: np.ndarray) -> int:
    """Hitung query yang hasilnya kurang dari k (ada slot −1).

    Args:
        indices: Indeks hasil per query.

    Returns:
        Jumlah query dengan paling sedikit satu slot −1.
    """
    return int((indices < 0).any(axis=1).sum())


def measure_latency_p50(index: faiss.Index, queries: np.ndarray, k: int, warmup_count: int, rounds: int) -> float:
    """Ukur latensi p50 pencarian satu query per panggilan (#10, 005a).

    Args:
        index: Index FAISS.
        queries: Vektor query split.
        k: Jumlah hasil per query.
        warmup_count: Jumlah query pemanasan dari awal split, tidak dihitung.
        rounds: Jumlah putaran atas semua query.

    Returns:
        Median waktu index.search per query dalam milidetik.
    """
    single_queries = [queries[row : row + 1] for row in range(len(queries))]
    for query in single_queries[:warmup_count]:
        index.search(query, k)
    timings = []
    for _ in range(rounds):
        for query in single_queries:
            start = time.perf_counter_ns()
            index.search(query, k)
            timings.append(time.perf_counter_ns() - start)
    return float(np.median(timings)) / 1e6


def measure_qps(index: faiss.Index, queries: np.ndarray, k: int, repeats: int) -> float:
    """Ukur QPS pencarian batch semua query (#9, 005a, 006a).

    Args:
        index: Index FAISS, dengan jumlah thread sudah diset.
        queries: Vektor query split.
        k: Jumlah hasil per query.
        repeats: Jumlah ulangan yang diukur setelah pemanasan T0.

    Returns:
        Jumlah query dibagi median waktu ulangan, dalam query per detik.
    """
    index.search(queries, k)
    timings = []
    for _ in range(repeats):
        start = time.perf_counter_ns()
        index.search(queries, k)
        timings.append(time.perf_counter_ns() - start)
    return len(queries) / (float(np.median(timings)) / 1e9)


def measure_index_size(index: faiss.Index) -> int:
    """Ukur ukuran index sebagai jumlah byte hasil serialisasi (#12, 005a).

    Args:
        index: Index FAISS.

    Returns:
        Jumlah byte faiss.serialize_index(index).
    """
    buffer = faiss.serialize_index(index)
    size = len(buffer)
    del buffer
    return size


def evaluate_index(
    index: faiss.Index,
    queries: np.ndarray,
    query_ids: np.ndarray,
    exact_indices: np.ndarray,
    exact_distances: np.ndarray,
    build_seconds: float,
) -> tuple[dict[str, float | int], dict[str, int]]:
    """Nilai dan ukur satu konfigurasi index pada query split.

    Args:
        index: Index FAISS dengan parameter search konfigurasi ini sudah diset.
        queries: Vektor query split.
        query_ids: query_id per baris queries.
        exact_indices: Indeks Tetangga exact.
        exact_distances: Jarak L2 Tetangga exact.
        build_seconds: Waktu build index (#11).

    Returns:
        12 metrik K7 dalam urutan kolom catatan run, dan kolom diagnostik.
    """
    scores, indices = search_index(index, queries, K)
    distances = to_result_distances(to_distances(scores), indices, MISSING_RESULT_DISTANCE)
    quality = compute_quality_metrics(to_doc_ids(indices, doc_ids), query_ids, relevant, K)
    fidelity = compute_fidelity_metrics(indices, distances, exact_indices, exact_distances, NEAR_ZERO_DISTANCE)
    latency_p50_ms = measure_latency_p50(index, queries, K, LATENCY_WARMUP_QUERIES, LATENCY_ROUNDS)
    qps = measure_qps(index, queries, K, QPS_REPEATS)
    metrics = {
        **quality,
        "knn_recall_at_5": fidelity["knn_recall_at_5"],
        "relative_distance_error": fidelity["relative_distance_error"],
        "qps": qps,
        "latency_p50_ms": latency_p50_ms,
        "build_seconds": build_seconds,
        "index_size_bytes": measure_index_size(index),
    }
    diagnostics = {
        "short_result_queries": count_short_results(indices),
        "rde_excluded_terms": fidelity["rde_excluded_terms"],
        "rde_queries_without_terms": fidelity["rde_queries_without_terms"],
    }
    return metrics, diagnostics

## 5. Sapuan nprobe

Satu konfigurasi per nilai nprobe pada index yang sama; nprobe diset sebelum pencarian dan pemanasan T₀ konfigurasi itu (006a).

In [ ]:
exact_distances = to_distances(exact_scores)
results = []
for nprobe in IVF_NPROBE_VALUES:
    index.nprobe = nprobe
    metrics, diagnostics = evaluate_index(index, split_vectors, split_query_ids, exact_indices, exact_distances,
                                          build_seconds)
    params = {"build": {"nlist": IVF_NLIST, **train_info}, "search": {"nprobe": nprobe}}
    results.append((params, metrics, diagnostics))
    print({"nprobe": nprobe, **metrics, **diagnostics})

## 6. Lingkungan

Fungsi disalin identik dari `04a_hnsw`; env_id harus sama dengan `03_exact` kalau dijalankan di instance yang sama (K8, 004a).

In [ ]:
import hashlib
import importlib.metadata
import platform
import resource
import shutil
import socket
import subprocess
from datetime import datetime, timezone


def parse_nvidia_smi(text: str) -> list[dict[str, str | int]]:
    """Ubah keluaran nvidia-smi menjadi daftar GPU.

    Args:
        text: Keluaran `nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader,nounits`,
            satu baris per GPU, contoh "NVIDIA GeForce RTX 3090, 24576, 550.54.14".

    Returns:
        Per GPU: nama, VRAM total (MiB), versi driver.
    """
    gpus = []
    for line in text.strip().splitlines():
        name, memory_total, driver = (part.strip() for part in line.rsplit(",", 2))
        gpus.append({"name": name, "memory_total_mib": int(memory_total), "driver": driver})
    return gpus


def fetch_gpu_info() -> list[dict[str, str | int]]:
    """Baca model GPU, VRAM total, dan driver lewat nvidia-smi.

    Returns:
        Daftar GPU.
    """
    command = ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"]
    return parse_nvidia_smi(subprocess.run(command, capture_output=True, text=True, check=True).stdout)


def parse_cpuinfo(text: str) -> dict[str, str | bool]:
    """Ambil model CPU dan flag AVX dari isi /proc/cpuinfo.

    Args:
        text: Isi /proc/cpuinfo.

    Returns:
        Model CPU, ada tidaknya flag avx2 dan avx512f.

    Raises:
        ValueError: Baris "model name" atau "flags" tidak ada.
    """
    fields = {}
    for line in text.splitlines():
        key, _, value = line.partition(":")
        fields.setdefault(key.strip(), value.strip())
    if "model name" not in fields or "flags" not in fields:
        raise ValueError("/proc/cpuinfo tidak memuat 'model name' atau 'flags'")
    flags = set(fields["flags"].split())
    return {"model": fields["model name"], "avx2": "avx2" in flags, "avx512f": "avx512f" in flags}


def parse_memory_max(text: str) -> int | None:
    """Ubah isi cgroup v2 `memory.max` menjadi byte.

    Args:
        text: Isi berkas, contoh "34359738368" atau "max".

    Returns:
        Jatah RAM dalam byte, atau None kalau tanpa batas.
    """
    value = text.strip()
    return None if value == "max" else int(value)


def parse_memory_limit(text: str, unlimited_from: int) -> int | None:
    """Ubah isi cgroup v1 `memory.limit_in_bytes` menjadi byte.

    Args:
        text: Isi berkas, contoh "34359738368", atau angka sekitar 2^63 untuk tanpa batas.
        unlimited_from: Nilai mulai dari angka ini dianggap tanpa batas.

    Returns:
        Jatah RAM dalam byte, atau None kalau tanpa batas.
    """
    value = int(text.strip())
    return None if value >= unlimited_from else value


def fetch_memory_quota(cgroup_dir: Path, unlimited_from: int) -> int | None:
    """Baca jatah RAM dari cgroup v2 atau v1.

    Args:
        cgroup_dir: Folder induk cgroup.
        unlimited_from: Batas v1 yang dianggap tanpa batas.

    Returns:
        Jatah RAM dalam byte, atau None kalau tanpa batas.

    Raises:
        ValueError: Berkas v2 dan v1 sama-sama ada (host hybrid, urutan deteksi belum diputuskan, H18).
        FileNotFoundError: Berkas v2 maupun v1 tidak ada.
    """
    v2_path = cgroup_dir / "memory.max"
    v1_path = cgroup_dir / "memory" / "memory.limit_in_bytes"
    has_v2, has_v1 = v2_path.is_file(), v1_path.is_file()
    if has_v2 and has_v1:
        raise ValueError(f"{v2_path} dan {v1_path} sama-sama ada; urutan deteksi host hybrid belum diputuskan (H18)")
    if has_v2:
        return parse_memory_max(v2_path.read_text(encoding="utf-8"))
    if has_v1:
        return parse_memory_limit(v1_path.read_text(encoding="utf-8"), unlimited_from)
    raise FileNotFoundError(f"Tidak ada {v2_path} maupun {v1_path}; butuh cgroup v1 atau v2 di Linux (H12)")


def fetch_library_versions(names: list[str]) -> dict[str, str]:
    """Baca versi paket terpasang tanpa meng-import paketnya.

    Args:
        names: Nama distribusi paket.

    Returns:
        Nama paket ke versi.
    """
    return {name: importlib.metadata.version(name) for name in names}


def collect_environment(
    cpu_quota: float, revisions: dict[str, object], project_root: Path, cgroup_dir: Path, cpuinfo_path: Path
) -> dict[str, dict[str, object]]:
    """Kumpulkan field statis dan dinamis Lingkungan (K8, 004a).

    Args:
        cpu_quota: Jatah CPU efektif dari fetch_cpu_quota.
        revisions: Revision model dan dataset.
        project_root: Root project untuk ukuran disk.
        cgroup_dir: Folder cgroup v2.
        cpuinfo_path: Berkas /proc/cpuinfo.

    Returns:
        {"static": field yang membentuk env_id, "dynamic": field yang hanya dicatat}.
    """
    disk = shutil.disk_usage(project_root)
    static = {
        "gpus": fetch_gpu_info(),
        "cuda": torch.version.cuda,
        "cpu": parse_cpuinfo(cpuinfo_path.read_text(encoding="utf-8")),
        "cpu_quota": cpu_quota,
        "cpu_total": os.cpu_count(),
        "memory_quota_bytes": fetch_memory_quota(cgroup_dir, MEMORY_UNLIMITED_FROM),
        "disk_total_bytes": disk.total,
        "threads_faiss": faiss.omp_get_max_threads(),
        "threads_torch": torch.get_num_threads(),
        "python": platform.python_version(),
        "libraries": fetch_library_versions(ENVIRONMENT_LIBRARIES),
        "revisions": revisions,
        "hostname": socket.gethostname(),
    }
    dynamic = {
        # ru_maxrss di Linux dalam KiB
        "peak_ram_bytes": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024,
        "peak_vram_bytes": torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None,
        "disk_free_bytes": disk.free,
        # Cap waktu UTC format ISO 8601, contoh 2026-10-02T08:30:00+00:00
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    return {"static": static, "dynamic": dynamic}


def compute_env_id(static: dict[str, object], length: int) -> str:
    """Hitung env_id dari field statis Lingkungan.

    Args:
        static: Field statis, termasuk hostname.
        length: Jumlah karakter heksadesimal yang diambil dari awal hash.

    Returns:
        Awal SHA-256 dari JSON field statis berkunci terurut.
    """
    canonical = json.dumps(static, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:length]


def save_environment(environment: dict[str, dict[str, object]], env_id: str, notebook: str, tuning_dir: Path) -> Path:
    """Tambahkan catatan Lingkungan ke `environment_<env_id>.json`.

    Args:
        environment: Hasil collect_environment.
        env_id: Hasil compute_env_id.
        notebook: Nama notebook pencatat.
        tuning_dir: Folder outputs/tuning.

    Returns:
        Path berkas Lingkungan.

    Raises:
        ValueError: Berkas env_id yang sama sudah ada tetapi field statisnya berbeda.
    """
    path = tuning_dir / f"environment_{env_id}.json"
    record = {"notebook": notebook, **environment["dynamic"]}
    if path.is_file():
        with path.open(encoding="utf-8") as file:
            content = json.load(file)
        if content["static"] != json.loads(json.dumps(environment["static"])):
            raise ValueError(f"{path}: field statis berbeda untuk env_id yang sama")
        content["records"].append(record)
    else:
        content = {"env_id": env_id, "static": environment["static"], "records": [record]}
    tuning_dir.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as file:
        json.dump(content, file, ensure_ascii=False, indent=2)
    return path


embedding_config = embedding_set["config"]
revisions = {"model": embedding_config["model_revision"], "datasets": embedding_config["dataset_revisions"]}
environment = collect_environment(cpu_quota, revisions, PROJECT_ROOT, CGROUP_DIR, CPUINFO_PATH)
env_id = compute_env_id(environment["static"], ENV_ID_LENGTH)
environment_path = save_environment(environment, env_id, NOTEBOOK_NAME, TUNING_DIR)
print({"env_id": env_id, "path": str(environment_path), **environment["dynamic"]})

## 7. Catat run

Satu baris per nprobe (K4) di `runs_ivf.csv`; berkas hanya ditambah barisnya. Fungsi disalin identik dari `04a_hnsw`.

In [ ]:
import uuid
from datetime import datetime, timezone


def build_run(
    algorithm: str,
    params: dict[str, dict[str, object]],
    metrics: dict[str, float | int],
    diagnostics: dict[str, int],
    identity: dict[str, str],
) -> dict[str, object]:
    """Susun satu baris catatan run (K4).

    Args:
        algorithm: flat, hnsw, ivf, atau lsh.
        params: {"build": ..., "search": ...}; IVF juga jumlah sampel latih dan seed k-means, LSH seed rotasi.
        metrics: 12 metrik K7.
        diagnostics: Kolom diagnostik (hasil < 5, suku #8 dikeluarkan, query tanpa suku #8).
        identity: embedding_id, env_id, split.

    Returns:
        Baris run dengan run_id baru dan cap waktu.
    """
    return {
        "run_id": uuid.uuid4().hex,
        # Cap waktu UTC format ISO 8601, contoh 2026-10-02T08:30:00+00:00
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "embedding_id": identity["embedding_id"],
        "env_id": identity["env_id"],
        "algorithm": algorithm,
        "params": json.dumps(params, ensure_ascii=False, sort_keys=True),
        "split": identity["split"],
        "k": K,
        "threads_faiss": faiss.omp_get_max_threads(),
        "threads_torch": torch.get_num_threads(),
        **metrics,
        **diagnostics,
    }


def append_run(run: dict[str, object], path: Path) -> None:
    """Tambahkan satu baris run ke CSV tanpa menimpa baris lama.

    Args:
        run: Baris run.
        path: Berkas CSV catatan run.

    Raises:
        ValueError: Kolom berkas lama berbeda dengan kolom run.
    """
    row = pd.DataFrame([run])
    if path.is_file():
        existing_columns = pd.read_csv(path, nrows=0).columns.tolist()
        if existing_columns != row.columns.tolist():
            raise ValueError(f"Kolom {path} berbeda dengan kolom run baru; baris tidak ditambahkan")
        row.to_csv(path, mode="a", header=False, index=False, lineterminator="\n")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    row.to_csv(path, index=False, lineterminator="\n")


identity = {"embedding_id": embedding_id, "env_id": env_id, "split": SPLIT}
for params, metrics, diagnostics in results:
    run = build_run(ALGORITHM, params, metrics, diagnostics, identity)
    append_run(run, TUNING_DIR / f"runs_{ALGORITHM}.csv")
    print(run)